In [ ]:
# airline_data_cleaning.ipynb
# Data Cleaning and Processing
# Checking validity of year, month, airport codes, uniqueness of carrier codes and names
import pandas as pd

In [ ]:
df = pd.read_csv("Airline_Delay_Cause.csv") # load in raw dataset

In [ ]:
# invalid year not in YYYY format?
year_valid = df['year'].astype(str).str.match(r'^\d{4}$')
invalid_years = df[~year_valid]
print(len(invalid_years)==0) # True if all years are already valid

True


In [ ]:
# invalid month not within 1-12?
month_valid = df['month'].between(1, 12)
invalid_months = df[~month_valid]
print(len(invalid_months)==0) # True if all months are already valid

True


In [ ]:
# "airport" column should be 3 character alpha-numeric
valid_len = df['airport'].str.len() == 3
valid_ch = df['airport'].str.isalnum()
invalid_airport = df[~(valid_len & valid_ch)]
print(len(invalid_airport)==0) # True if all codes are already valid

True


In [ ]:
keys = ['year', 'month', 'carrier', 'airport']
duplicates = df[df.duplicated(subset=keys, keep=False)]
print(duplicates)
# empty: everything unique

Empty DataFrame
Columns: [year, month, carrier, carrier_name, airport, airport_name, arr_flights, arr_del15, carrier_ct, weather_ct, nas_ct, security_ct, late_aircraft_ct, arr_cancelled, arr_diverted, arr_delay, carrier_delay, weather_delay, nas_delay, security_delay, late_aircraft_delay]
Index: []

[0 rows x 21 columns]


In [ ]:
# check carrier and carrier_name are 1:1
carrier_to_name = df.groupby('carrier')['carrier_name'].unique()
carrier_to_name = carrier_to_name.map(lambda x: ", ".join(map(str, x)))
print(carrier_to_name)

carrier
9E            Endeavor Air Inc., Pinnacle Airlines Inc.
9K                                             Cape Air
AA    American Airlines Network, American Airlines Inc.
AQ                                  Aloha Airlines Inc.
AS        Alaska Airlines Network, Alaska Airlines Inc.
AX                                Trans States Airlines
B6                                      JetBlue Airways
C5    CommuteAir LLC dba CommuteAir, Commutair Aka C...
CO                           Continental Air Lines Inc.
CP                                     Compass Airlines
DH            Independence Air, Atlantic Coast Airlines
DL        Delta Air Lines Network, Delta Air Lines Inc.
EM                                 Empire Airlines Inc.
EV    ExpressJet Airlines LLC, ExpressJet Airlines I...
F9            Frontier Airlines, Frontier Airlines Inc.
FL                          AirTran Airways Corporation
G4                                        Allegiant Air
G7              GoJet Airlines LLC d/b/a

Above we see that some (many) carrier codes have multiple carrier names associated with them. Some are very similar, like "AA" having "American Airlines Network" and "American Airlines Inc." However, there are a few with completely different names, like "9E" with "Endeavor Air Inc." and "Pinnacle Airlines Inc."

In [ ]:
carrier_to_name.to_csv("carrier_to_name.csv") # save codes with their lists of names to a CSV

In [ ]:
name_to_carrier = df.groupby('carrier_name')['carrier'].unique()
name_to_carrier = name_to_carrier.map(lambda x: ", ".join(map(str, x)))
print(name_to_carrier)

carrier_name
ATA Airlines d/b/a ATA                               TZ
Air Wisconsin Airlines Corp                          ZW
AirTran Airways Corporation                          FL
Alaska Airlines Inc.                                 AS
Alaska Airlines Network                              AS
Allegiant Air                                        G4
Aloha Airlines Inc.                                  AQ
America West Airlines Inc.                           HP
American Airlines Inc.                               AA
American Airlines Network                            AA
American Eagle Airlines Inc.                         MQ
Atlantic Coast Airlines                              DH
Atlantic Southeast Airlines                          EV
Cape Air                                             9K
Comair Inc.                                          OH
Commutair Aka Champlain Enterprises, Inc.            C5
CommuteAir LLC dba CommuteAir                        C5
Compass Airlines                   

In [ ]:
name_to_carrier.to_csv("name_to_carrier.csv") # save carrier names with their code(s) to a CSV

In [ ]:
# check for null values in any of the columns
print(df.isnull().sum())

year                     0
month                    0
carrier                  0
carrier_name             0
airport                  0
airport_name             0
arr_flights            662
arr_del15              960
carrier_ct             662
weather_ct             662
nas_ct                 662
security_ct            662
late_aircraft_ct       662
arr_cancelled          662
arr_diverted           662
arr_delay              662
carrier_delay          662
weather_delay          662
nas_delay              662
security_delay         662
late_aircraft_delay    662
dtype: int64


In [ ]:
# drop 662 columns where flight data for given carrier at airport was not present
df_dropped = df.dropna(subset=['arr_flights'])
# fill the null fields with 0
df_dropped['arr_del15'] = df_dropped['arr_del15'].fillna(0)
df_dropped.to_csv("airline_delay_cleaned.csv", index=False) # save cleaned dataset

In [ ]:
# separate the ExpressJet Airlines Inc. carrier codes (EV, XE, RU) into 3 different names
name = "ExpressJet Airlines Inc."
# add on -[code] to name
df2 = df_dropped.copy()
df2.loc[df2['carrier_name']==name, 'carrier_name'] = df2['carrier_name'] + "-" + df2['carrier']
# save to new csv
df2.to_csv("airline_delay_cleaned2.csv", index=False) # final cleaned dataset